<a href="https://colab.research.google.com/github/nashwaparvis/DBA-NorthStar-Coursework/blob/main/01_SQL_in_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
system("git clone https://github.com/nashwaparvis/DBA-NorthStar-Coursework.git")

In [ ]:
list.files()

[1] "DBA-NorthStar-Coursework" "sample_data"

In [ ]:
install.packages("sqldf")
install.packages("readr")
install.packages("dplyr")
install.packages("ggplot2")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)



In [ ]:
library(sqldf)
library(readr)
library(dplyr)
library(ggplot2)

Loading required package: gsubfn

Loading required package: proto

Warning message:
“no DISPLAY variable so Tk is not available”
Loading required package: RSQLite


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [ ]:
drivers <- read_csv("DBA-NorthStar-Coursework/dataset/drivers.csv")
vehicles <- read_csv("DBA-NorthStar-Coursework/dataset/vehicles.csv")
hubs <- read_csv("DBA-NorthStar-Coursework/dataset/hubs.csv")
incidents <- read_csv("DBA-NorthStar-Coursework/dataset/incidents.csv")
customers <- read_csv("DBA-NorthStar-Coursework/dataset/customers.csv")
app_events <- read_csv("DBA-NorthStar-Coursework/dataset/app_events.csv")
orders <- read_csv("DBA-NorthStar-Coursework/dataset/orders.csv")
deliveries <- read_csv("DBA-NorthStar-Coursework/dataset/deliveries.csv")
complaints <- read_csv("DBA-NorthStar-Coursework/dataset/complaints.csv")

Rows: 170 Columns: 8
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (4): driver_id, base_zone, employment_type, shift_preference
dbl (4): years_experience, training_score, driver_rating, active_flag

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 120 Columns: 8
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (5): vehicle_id, vehicle_type, assigned_zone, maintenance_status, telem...
dbl  (2): battery_health_pct, odometer_km
dttm (1): commission_date

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 8 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (4): hub_id, hub_name, zone, hub_type
dbl (1): capacit

In [ ]:
head(orders)

order_id,customer_id,service_type,order_created_at,promised_window_hours,pickup_zone,dropoff_zone,priority_level,order_value,booking_channel,special_handling_flag
<chr>,<chr>,<chr>,<dttm>,<dbl>,<chr>,<chr>,<chr>,<dbl>,<chr>,<dbl>
O00001,C0292,Passenger,2024-08-20 14:43:00,6,Airport,South,Medium,126.65,App,0
O00002,C0459,Passenger,2024-05-14 22:16:00,24,North,AIRPORT,Low,109.30,App,0
O00003,C0161,Passenger,2025-09-02 14:37:00,4,West,AIRPORT,High,33.50,Phone,0
O00004,C0520,Parcel,2025-01-11 17:15:00,2,RiverSide,North,Medium,10.04,App,1
O00005,C0558,Retail,2025-02-17 19:32:00,12,Riverside,SOUTH,Low,125.58,Phone,0
O00006,C0437,Retail,2024-08-05 04:55:00,1,CENTRAL,East,High,151.44,Web,1


In [ ]:
head(deliveries)

delivery_id,order_id,driver_id,vehicle_id,hub_id,dispatch_time,delivery_completed_at,delivery_status,route_distance_km,manual_route_override_count,proof_of_completion_missing,customer_rating_post_delivery,fuel_or_charge_cost
<chr>,<chr>,<chr>,<chr>,<chr>,<dttm>,<dttm>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
DL00001,O00938,D004,V056,H05,2024-06-18 10:57:00,2024-06-19 09:05:59,Failed,17.26,1,0,3.07,12.05
DL00002,O00004,D138,V007,H02,2025-01-11 18:45:00,2025-01-11 17:39:00,OnTime,10.34,1,0,5.00,13.41
DL00003,O00639,D006,V049,H02,2025-06-02 20:39:00,2025-06-02 21:45:32,OnTime,7.92,0,0,4.98,8.51
DL00004,O00313,D116,V055,H02,2024-03-08 23:31:00,2024-03-09 23:30:08,Delayed,16.42,0,0,4.18,13.62
DL00005,O00844,D108,V034,H01,2025-09-21 11:43:00,2025-09-21 15:45:34,OnTime,14.52,1,0,4.18,9.22
DL00006,O00029,D037,V098,H03,2024-09-11 12:40:00,2024-09-12 17:11:52,Delayed,13.84,0,0,1.57,9.58


In [ ]:
head(complaints)

complaint_id,customer_id,order_id,complaint_type,channel,severity,created_at,status,resolution_days,compensation_amount
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dttm>,<chr>,<dbl>,<dbl>
CP0001,C0464,O00814,AppIssue,App,High,2025-03-30 02:36:00,Open,11,23.99
CP0002,C0056,O00628,MissedPickup,Phone,Medium,2024-11-07 10:05:00,Open,4,21.64
CP0003,C0469,O00384,Delay,Chatbot,High,2024-01-02 15:47:00,Open,16,26.41
CP0004,C0631,O00406,Delay,App,Medium,2025-01-14 13:07:00,AwaitingCustomer,7,23.44
CP0005,C0535,O00154,Delay,Email,Medium,2024-08-31 05:56:00,Resolved,1,16.18
CP0006,C0096,O00147,Delay,App,Medium,2024-07-22 07:43:00,Resolved,9,18.51


In [ ]:
sqldf("
SELECT
  delivery_status,
  COUNT(*) AS total_deliveries
FROM deliveries
GROUP BY delivery_status
ORDER BY total_deliveries DESC
")

delivery_status,total_deliveries
<chr>,<int>
OnTime,616
Delayed,202
Failed,132


In [ ]:
sqldf("
SELECT
  h.zone,
  COUNT(d.delivery_id) AS total_deliveries,
  AVG(d.manual_route_override_count) AS avg_route_overrides,
  AVG(d.customer_rating_post_delivery) AS avg_rating
FROM deliveries d
JOIN hubs h
ON d.hub_id = h.hub_id
GROUP BY h.zone
ORDER BY avg_route_overrides DESC
")

zone,total_deliveries,avg_route_overrides,avg_rating
<chr>,<int>,<dbl>,<dbl>
Riverside,115,1.0521739,3.881858
Central,243,1.0329218,3.782479
North,136,1.0294118,3.840593
South,106,0.9150943,3.950952
Airport,104,0.9134615,3.882136
East,119,0.8907563,3.895862
West,127,0.8740157,3.915476
